# Featherstore synthetic recommendation demo

Start the Rust server first with `cargo run -p featherstore-server --release`, then run the cells below. Install the SDK with `python -m pip install -e ./python[dev]`.


In [ ]:
from featherstore import FeatherstoreClient, LookupRequest, generate_recommendation_dataset
SERVER_URL = 'http://localhost:8080'
dataset = generate_recommendation_dataset(users=100, items=100, contexts=24, embedding_dims=4, seed=13)
dataset.user_schema, dataset.item_schema, dataset.context_schema


In [ ]:
client = FeatherstoreClient(SERVER_URL)
client.health()


In [ ]:
for rows in dataset.iter_user_batches(25):
    client.ingest_batch(dataset.user_schema, rows)
for rows in dataset.iter_item_batches(25):
    client.ingest_batch(dataset.item_schema, rows)
for rows in dataset.iter_context_batches(24):
    client.ingest_batch(dataset.context_schema, rows)
client.ready()


In [ ]:
user = client.get_features('user', '42', ['age_bucket', 'country_id', 'u_emb_0'])
item = client.get_features('item', '42', ['category_id', 'price_bucket', 'i_emb_0'])
context = client.get_features('context', '12', ['hour_bucket', 'device_type', 'c_emb_0'])
user, item, context


In [ ]:
batch = client.lookup([
    LookupRequest('user', '7', ['age_bucket', 'country_id']),
    LookupRequest('item', '7', ['category_id', 'price_bucket']),
    LookupRequest('context', '7', ['hour_bucket', 'device_type']),
])
batch


## Benchmark smoke command

This is a smoke check, not proof of the 10k QPS p99 < 5 ms target.


In [ ]:
print('featherstore-benchmark --server http://localhost:8080 --scenario single_get_request_qps --duration 5 --warmup 1 --concurrency 16 --target-qps 1000 --users 100 --items 100 --contexts 24 --embedding-dims 4 --output benchmark-smoke.json')


In [ ]:
client.close()
